# 1. Audit, Log-Transform e Preprocessing

Questo notebook costruisce la base dati finale del progetto. Le scelte chiave sono:

- esclusione di identificativi, coordinate e categorie derivate dal core modelling;
- mantenimento delle label derivate solo per interpretazione post-hoc;
- trasformazione `log1p` delle variabili astronomiche con code molto lunghe;
- imputazione mediana e standardizzazione per i metodi distance-based.


In [1]:
from pathlib import Path
import os

if Path.cwd().name != "notebook_final" and (Path.cwd() / "notebook_final").exists():
    os.chdir(Path.cwd() / "notebook_final")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "input" / "nasa_exoplanet_intelligence.csv"

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

SEED = 42
df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset raw: {df.shape[0]} righe, {df.shape[1]} colonne")
print("Colonne:")
print(list(df.columns))
display(df.head())


Dataset raw: 6150 righe, 31 colonne
Colonne:
['planet_name', 'host_star', 'n_stars', 'n_planets', 'discovery_method', 'disc_year', 'disc_facility', 'orbital_period_days', 'planet_radius_earth', 'planet_mass_earth', 'equilibrium_temp_k', 'orbital_eccentricity', 'semi_major_axis_au', 'star_temp_k', 'star_radius_sun', 'star_mass_sun', 'star_age_gyr', 'star_surface_gravity', 'star_metallicity', 'dist_from_earth_pc', 'star_vmag', 'ra', 'dec', 'controversial_flag', 'planet_type', 'habitable_zone_flag', 'multi_planet_system', 'is_recent_discovery', 'dist_category', 'star_type', 'orbital_period_cat']


,planet_name,host_star,n_stars,n_planets,discovery_method,disc_year,disc_facility,orbital_period_days,planet_radius_earth,planet_mass_earth,equilibrium_temp_k,orbital_eccentricity,semi_major_axis_au,star_temp_k,star_radius_sun,star_mass_sun,star_age_gyr,star_surface_gravity,star_metallicity,dist_from_earth_pc,star_vmag,ra,dec,controversial_flag,planet_type,habitable_zone_flag,multi_planet_system,is_recent_discovery,dist_category,star_type,orbital_period_cat
0,Kepler-1167 b,Kepler-1167,1,1,Transit,2016.0,Kepler,1.003934,1.710000,3.570,1419.0,0.0,0.01750,4971.0,0.750,0.790,4.27,4.600,-0.05,820.905,16.0470,298.302660,47.693965,0,Super-Earth,False,False,False,Far(500-2kpc),K-type,Short(1-10d)
1,Kepler-1740 b,Kepler-1740,1,1,Transit,2021.0,Kepler,8.172400,3.323214,11.000,858.0,0.0,0.07790,5705.0,0.905,0.943,NaN,4.499,-0.06,1061.770,15.4540,293.873663,38.922455,0,Mini-Neptune,False,False,True,Far(500-2kpc),G-type(Sun-like),Short(1-10d)
2,Kepler-1581 b,Kepler-1581,1,1,Transit,2016.0,Kepler,6.283855,0.800000,0.437,1108.0,0.0,0.06865,6022.0,1.230,1.120,4.17,4.310,0.07,493.175,12.4420,287.371320,39.603623,0,Sub-Earth,False,False,False,Mid(100-500pc),F-type,Short(1-10d)
3,Kepler-644 b,Kepler-644,1,1,Transit,2016.0,Kepler,3.173917,3.150000,10.100,1655.0,0.0,0.04641,6747.0,1.810,1.490,1.62,4.090,0.08,1318.050,14.0310,295.475702,43.493112,0,Mini-Neptune,False,False,False,Far(500-2kpc),F-type,Short(1-10d)
4,Kepler-1752 b,Kepler-1752,1,1,Transit,2021.0,Kepler,56.358501,4.540605,18.700,419.0,0.0,0.26980,5446.0,0.821,0.824,7.20,4.525,-0.20,962.888,16.0249,290.854140,51.222743,0,Neptune-like,False,False,True,Far(500-2kpc),G-type(Sun-like),Medium(10-100d)


## 1.1 Audit dei valori mancanti e delle categorie

Prima dei modelli controlliamo missing values, tipi e distribuzioni delle variabili categoriche principali. Questo serve a distinguere feature fisiche, feature osservative, identificativi e label derivate.


In [2]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
}).sort_values("missing_pct", ascending=False)

print("Missing values > 0:")
display(missing[missing["missing_count"] > 0])

categorical_cols = [
    "discovery_method",
    "disc_facility",
    "planet_type",
    "habitable_zone_flag",
    "dist_category",
    "star_type",
    "orbital_period_cat",
]
for col in categorical_cols:
    if col in df.columns:
        print(f"\n=== {col} ===")
        display(df[col].value_counts(dropna=False).to_frame("count").head(15))


Missing values > 0:


,missing_count,missing_pct,dtype
equilibrium_temp_k,1563,25.41,float64
star_age_gyr,1311,21.32,float64
orbital_eccentricity,938,15.25,float64
star_metallicity,550,8.94,float64
orbital_period_days,334,5.43,float64
star_surface_gravity,318,5.17,float64
semi_major_axis_au,316,5.14,float64
star_radius_sun,314,5.11,float64
star_vmag,295,4.80,float64
star_temp_k,290,4.72,float64



=== discovery_method ===


,count
discovery_method,
Transit,4517
Radial Velocity,1182
Microlensing,275
Imaging,94
Transit Timing Variations,39
Eclipse Timing Variations,17
Orbital Brightness Modulation,9
Pulsar Timing,8
Astrometry,6



=== disc_facility ===


...

,count
dist_category,
Far(500-2kpc),2252
Mid(100-500pc),2029
Nearby(<100pc),1524
Distant(2k+pc),318
Unknown,27



=== star_type ===


,count
star_type,
G-type(Sun-like),2669
K-type,1644
F-type,1075
M-type(Red Dwarf),423
Unknown,290
A-type,25
B-type,19
O-type,5



=== orbital_period_cat ===


,count
orbital_period_cat,
Short(1-10d),2588
Medium(10-100d),2083
Very-Long(365d+),642
Long(100-365d),346
Unknown,334
Ultra-Short(<1d),157


## 1.2 Feature strategy

Per il clustering usiamo solo feature fisiche/orbitali/stellari numeriche. Non usiamo:

- `planet_type`, `star_type`, `orbital_period_cat`, `dist_category`, `habitable_zone_flag`, perche' sono label o categorie derivate;
- `planet_name`, `host_star`, `discovery_method`, `disc_facility`, perche' sono identificativi o metadati;
- `ra`, `dec`, perche' sono coordinate osservative, non famiglie fisiche;
- `disc_year`, `is_recent_discovery`, `controversial_flag`, perche' descrivono il processo di scoperta piu' che il pianeta.

Per l'estensione supervisionata includiamo alcune feature di contesto (`n_stars`, `n_planets`, `multi_planet_system`, distanza) che possono aiutare a misurare quanto resta predicibile `planet_type` dopo l'ablation.

**Nota su `star_vmag`**: la magnitudine visuale della stella e' esclusa dalle feature supervisionate. `star_vmag` e' una misura osservativa (la luminosita' apparente dipende fortemente dalla distanza dalla Terra, correlazione con `dist_from_earth_pc_log` = 0.69) e non rappresenta una proprieta' fisica intrinseca del sistema planetario. Includerla avrebbe introdotto informazione di bias osservativo, non informazione fisica.


In [3]:
LABEL_AND_DERIVED_COLS = [
    "planet_type",
    "star_type",
    "orbital_period_cat",
    "dist_category",
    "habitable_zone_flag",
]

IDENTIFIER_AND_META_COLS = [
    "planet_name",
    "host_star",
    "discovery_method",
    "disc_facility",
    "ra",
    "dec",
    "disc_year",
    "is_recent_discovery",
    "controversial_flag",
]

LOG_TRANSFORM_COLS = [
    "orbital_period_days",
    "planet_radius_earth",
    "planet_mass_earth",
    "semi_major_axis_au",
    "dist_from_earth_pc",
]

PLANETARY_ORBITAL_FEATURES = [
    "equilibrium_temp_k",
    "orbital_eccentricity",
    "orbital_period_days_log",
    "planet_radius_earth_log",
    "planet_mass_earth_log",
    "semi_major_axis_au_log",
]

STELLAR_FEATURES = [
    "star_temp_k",
    "star_radius_sun",
    "star_mass_sun",
    "star_age_gyr",
    "star_surface_gravity",
    "star_metallicity",
]

SUPERVISED_CONTEXT_FEATURES = [
    "n_stars",
    "n_planets",
    "multi_planet_system",
    "dist_from_earth_pc_log",
]

print("Feature clustering planetarie/orbitali:")
print(PLANETARY_ORBITAL_FEATURES)
print("\nFeature stellari:")
print(STELLAR_FEATURES)
print("\nFeature contestuali supervised (senza star_vmag: osservativa, non fisica):")
print(SUPERVISED_CONTEXT_FEATURES)


Feature clustering planetarie/orbitali:
['equilibrium_temp_k', 'orbital_eccentricity', 'orbital_period_days_log', 'planet_radius_earth_log', 'planet_mass_earth_log', 'semi_major_axis_au_log']

Feature stellari:
['star_temp_k', 'star_radius_sun', 'star_mass_sun', 'star_age_gyr', 'star_surface_gravity', 'star_metallicity']

Feature contestuali supervised (senza star_vmag: osservativa, non fisica):
['n_stars', 'n_planets', 'multi_planet_system', 'dist_from_earth_pc_log']


## 1.3 Log-transform

Le variabili astronomiche hanno ordini di grandezza molto diversi. La `log1p` riduce l'effetto degli outlier estremi sulle distanze euclidee usate dai metodi di clustering distance-based.


In [4]:
df_master = df.copy()

bool_cols = df_master.select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    df_master[col] = df_master[col].astype(int)

for col in LOG_TRANSFORM_COLS:
    if col in df_master.columns:
        df_master[col + "_log"] = np.log1p(df_master[col].clip(lower=0))
        df_master = df_master.drop(columns=[col])

print("Colonne trasformate:")
print([c + "_log" for c in LOG_TRANSFORM_COLS])

fig, axes = plt.subplots(len(LOG_TRANSFORM_COLS), 2, figsize=(12, 3 * len(LOG_TRANSFORM_COLS)))
for i, col in enumerate(LOG_TRANSFORM_COLS):
    if col not in df.columns:
        continue
    axes[i, 0].hist(df[col].dropna(), bins=50, color="steelblue", edgecolor="white")
    axes[i, 0].set_title(f"{col} originale")
    axes[i, 1].hist(np.log1p(df[col].clip(lower=0).dropna()), bins=50, color="darkorange", edgecolor="white")
    axes[i, 1].set_title(f"{col}_log")
plt.tight_layout()
plt.show()


Colonne trasformate:
['orbital_period_days_log', 'planet_radius_earth_log', 'planet_mass_earth_log', 'semi_major_axis_au_log', 'dist_from_earth_pc_log']


## 1.4 Imputazione e scaling

`X_scaled` e' la matrice numerica standardizzata usata da EDA e clustering. La parte supervisionata usera' una pipeline con imputazione e scaling interni alla cross-validation, per evitare leakage dal test set.


In [5]:
MODELING_FEATURES = (
    PLANETARY_ORBITAL_FEATURES
    + STELLAR_FEATURES
    + SUPERVISED_CONTEXT_FEATURES
)
MODELING_FEATURES = [c for c in MODELING_FEATURES if c in df_master.columns]

X_raw = df_master[MODELING_FEATURES].copy()

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=MODELING_FEATURES, index=df_master.index)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=MODELING_FEATURES, index=df_master.index)

print(f"Feature modellistiche: {len(MODELING_FEATURES)}")
print(MODELING_FEATURES)
print(f"Null residui dopo imputazione: {int(X_imputed.isna().sum().sum())}")
display(X_scaled.describe().round(3))


Feature modellistiche: 16
['equilibrium_temp_k', 'orbital_eccentricity', 'orbital_period_days_log', 'planet_radius_earth_log', 'planet_mass_earth_log', 'semi_major_axis_au_log', 'star_temp_k', 'star_radius_sun', 'star_mass_sun', 'star_age_gyr', 'star_surface_gravity', 'star_metallicity', 'n_stars', 'n_planets', 'multi_planet_system', 'dist_from_earth_pc_log']
Null residui dopo imputazione: 0


,equilibrium_temp_k,orbital_eccentricity,orbital_period_days_log,planet_radius_earth_log,planet_mass_earth_log,semi_major_axis_au_log,star_temp_k,star_radius_sun,star_mass_sun,star_age_gyr,star_surface_gravity,star_metallicity,n_stars,n_planets,multi_planet_system,dist_from_earth_pc_log
count,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000,6150.000
mean,0.000,0.000,0.000,-0.000,-0.000,0.000,-0.000,0.000,0.000,0.000,0.000,-0.000,-0.000,-0.000,-0.000,-0.000
std,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
min,-2.121,-0.468,-1.513,-1.848,-1.464,-0.456,-2.925,-0.380,-2.265,-1.611,-8.946,-5.654,-0.302,-0.669,-0.861,-3.255
25%,-0.584,-0.468,-0.671,-0.805,-0.744,-0.394,-0.273,-0.181,-0.407,-0.554,-0.204,-0.479,-0.302,-0.669,-0.861,-0.692
50%,-0.176,-0.468,-0.271,-0.392,-0.446,-0.338,0.083,-0.136,0.008,-0.119,0.152,0.022,-0.302,-0.669,-0.861,0.189
75%,0.325,-0.010,0.298,1.249,0.839,-0.149,0.280,-0.065,0.369,0.226,0.424,0.579,-0.302,0.195,1.162,0.724
max,7.843,6.149,8.664,3.851,2.564,12.030,30.245,22.599,24.430,4.426,8.564,3.250,8.442,5.375,1.162,2.292


## 1.5 Salvataggio artefatti


In [6]:
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df_master.to_csv(processed_dir / "df_master.csv", index=False)
X_raw.to_csv(processed_dir / "X_raw_modeling.csv", index=False)
X_imputed.to_csv(processed_dir / "X_imputed.csv", index=False)
X_scaled.to_csv(processed_dir / "X_scaled.csv", index=False)

metadata = {
    "n_samples": int(df_master.shape[0]),
    "n_raw_columns": int(df.shape[1]),
    "label_and_derived_cols_excluded_from_modeling": LABEL_AND_DERIVED_COLS,
    "identifier_and_meta_cols_excluded_from_clustering": IDENTIFIER_AND_META_COLS,
    "log_transform_cols_original": LOG_TRANSFORM_COLS,
    "log_transform_cols_final": [c + "_log" for c in LOG_TRANSFORM_COLS if c in df.columns],
    "planetary_orbital_features": PLANETARY_ORBITAL_FEATURES,
    "stellar_features": STELLAR_FEATURES,
    "supervised_context_features": SUPERVISED_CONTEXT_FEATURES,
    "modeling_features": MODELING_FEATURES,
    "imputer_strategy": "median",
    "scaler": "StandardScaler",
    "notes": [
        "habitable_zone_flag is kept only for post-hoc interpretation, never as a clustering feature",
        "supervised notebooks fit imputer/scaler inside CV pipelines to avoid test leakage",
        "star_vmag excluded from supervised features: observational bias (corr=0.69 with dist_from_earth_pc_log), not intrinsic planet system property",
    ],
}

with open(processed_dir / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Artefatti salvati in data/processed/")
for p in sorted(processed_dir.glob("*")):
    if p.is_file():
        print("-", p)


Artefatti salvati in data/processed/
- data\processed\df_master.csv
- data\processed\preprocessing_metadata.json
- data\processed\X_imputed.csv
- data\processed\X_raw_modeling.csv
- data\processed\X_scaled.csv
